<center>Universidade Federal de Viçosa</center>
<center>Coordenadoria de Educação Aberta e a Distância</center>
<center>Inteligência Artificial e Computacional</center>
<center>ELT578 - Análise de Imagens e Visão Computacional</center>

**<center>AULA PRÁTICA 6 - Parte I: Técnicas de Segmentação de Imagem</center>**

***<center>Limiarização e detecção de bordas</center>***

**1) Montando o Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**2) Importando bibliotecas**

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

**3) Limiarização global**

In [ ]:
img=cv2.imread('/content/drive/MyDrive/ELT578/SEMANA_3/bob.jpg',0)
cv2_imshow(img)

In [ ]:
plt.hist(img.ravel(), bins=255, range=(0, 256))
plt.show()

In [ ]:
l, I1= cv2.threshold(img,245,255,cv2.THRESH_BINARY) #acima do limear é branco (255), abaixo é preto (zero)
l2,I2 = cv2.threshold(img,245,255,cv2.THRESH_BINARY_INV) #abaixo do limear é branco, acima é preto
l3,I3 = cv2.threshold(img,245,255,cv2.THRESH_TRUNC) #abaixo do limear ele repete o valor e acima ele não altera a imagem orignial
l4,I4 = cv2.threshold(img,245,255,cv2.THRESH_TOZERO) #abaixo do limear ele zera e acima mantem os valores originais
l5,I5 = cv2.threshold(img,245,255,cv2.THRESH_TOZERO_INV) #acima do limear ele zera e abaixo mantem os valores originais

#print('Limiar: ', l)
cv2_imshow(I1)

**4) Limiarização global: Otsu**



In [ ]:
# Threshold Otsu (automático)
t, I_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Limiar calculado pelo Otsu: {t}")
cv2_imshow(I_otsu)

**5) Limiarização adaptativa**

In [ ]:
# 1) carregar em tons de cinza
img = cv2.imread('/content/drive/MyDrive/ELT578/SEMANA_3/sudoku.jpg', cv2.IMREAD_GRAYSCALE)
#cv2_imshow(img)

# 2) reduzir ruído com mediana (janela ímpar)
img_blur = cv2.medianBlur(img, 5)

# 3) limiarização global (fixa)
# ret = limiar usado (127)
ret, th1 = cv2.threshold(img_blur, 127, 255, cv2.THRESH_BINARY)

# 4) limiarização adaptativa (média)
# blockSize precisa ser ímpar e > 1
th2 = cv2.adaptiveThreshold(img_blur, 255,
                            cv2.ADAPTIVE_THRESH_MEAN_C,
                            cv2.THRESH_BINARY,
                            11,   # blockSize (vizinhança)
                            2)    # C (ajuste fino)

# 5) limiarização adaptativa (gaussiana)
th3 = cv2.adaptiveThreshold(img_blur, 255,
                            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                            cv2.THRESH_BINARY,
                            11,   # blockSize
                            2)    # C
# 6) visualizar
titles = ['Original (grayscale)', 'Global (v = 127)',
          'Adaptativa - Média', 'Adaptativa - Gaussiana']
images = [img, th1, th2, th3]

plt.figure(figsize=(10, 8))
for i in range(4):
    plt.subplot(2, 2, i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.tight_layout()
plt.show()

**6) Segmentação por Detecção de Bordas**
*  Filtro passa-alta (domínio da frequência)

In [ ]:
# 1. Leitura da imagem em escala de cinza
img = cv2.imread('/content/drive/MyDrive/ELT578/SEMANA_2/quarter.tif', 0)

# 2. Pré-processamento: aplicar desfoque de médias (mais forte que GaussianBlur)
img_blur = cv2.blur(img, (5, 5))

# 3. Transformada de Fourier 2D
f = np.fft.fft2(img_blur)
fshift = np.fft.fftshift(f)

# 4. Criar máscara passa-alta
rows, cols = img.shape
crow, ccol = rows // 2, cols // 2
r = 50  # raio do círculo central (ajuste conforme necessário)

mask_high = np.ones((rows, cols), np.uint8)
cv2.circle(mask_high, (ccol, crow), r, 0, -1)  # zera as baixas frequências

# 5. Aplicar a máscara
fshift_high = fshift * mask_high

# 6. Transformada inversa
img_high = np.fft.ifft2(np.fft.ifftshift(fshift_high))
img_high = np.abs(img_high)

# 7. Normalização
img_high = cv2.normalize(img_high, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# 8. Limiarização para segmentar bordas
_, moeda_mask = cv2.threshold(img_high, 30, 255, cv2.THRESH_BINARY)

# 9. Remover pequenos ruídos (opcional)
kernel = np.ones((5, 5), np.uint8)
moeda_mask = cv2.morphologyEx(moeda_mask, cv2.MORPH_CLOSE, kernel)

# 10. Exibir resultados
plt.figure(figsize=(15, 6))

plt.subplot(1, 3, 1), plt.imshow(img, cmap='gray')
plt.title('Imagem Original'), plt.axis('off')

plt.subplot(1, 3, 2), plt.imshow(img_high, cmap='gray')
plt.title('Passa-Alta'), plt.axis('off')

plt.subplot(1, 3, 3), plt.imshow(moeda_mask, cmap='gray')
plt.title('Segmentação da Moeda'), plt.axis('off')

plt.show()


**DESAFIO**
* **Limiarização Global em Espaço RGB:** Segmentar a vaca da imagem cow.jpg aplicando um limiar fixo diretamente nos 3 canais RGB, sem converter para escala de cinza.